# Team: Alexandro Galvez-Vega and Victor Neyra
# Class: CAP 5625-042 Computational Foundations Of AI
# Professor: Michael DeGiorgio

# __________________________________________________________________
# __________________________________________________________________

In [ ]:
# Starting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

# Deliverables Are Located Below
# __________________________________________________________________
# __________________________________________________________________

# The previous assignment was used as a template
# Similar early logic exists
# Here, we are temporarily turning the csv data into a data frame

filename = "Credit_N400_p9.csv"
advertising_df = pd.read_csv(filename)

# This dataframe is quickly split between the features that will be used for x
x_features_df = advertising_df.copy(deep=True)

# Our goal is to determine credit card balance which is separated into its own df here
y_features_df = x_features_df.pop("Balance") 

# In this part the categorical columns are converted to allow them to be used for the upcoming algorithms
x_features_df = pd.get_dummies(x_features_df, columns=["Gender"], drop_first=True, dtype=int)
x_features_df = pd.get_dummies(x_features_df, columns=["Married"], drop_first=True, dtype=int)
x_features_df = pd.get_dummies(x_features_df, columns=["Student"], drop_first=True, dtype=int)

# This simply shows that the columns were converted and the original columns are gone
# Earlier the df was placed into a deep copy of the original information so the original form exists too
x_features_df.head()

# For the purposes of the assignment, we are using ridge regression
# which requires standardization of the data
y_standardized = y_features_df - y_features_df.mean()

# The TA pointed out that encoded features need to be standardized as well
X_standardized = (x_features_df - x_features_df.mean()) / x_features_df.std()

# This time the starting learning rate is 10 ^ -5
learning_rate_alpha = 10 ** -5

# This is the information of our penalty term lambda
tuning_parameter = 10 ** -2
tuning_parameters_set = [10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4, 10**5, 10**6]

k_fold = 5  

# We get the total number of observations involved
observations_N = len(X_standardized)

# We note the total number of features too
feature_total_p = len(X_standardized.columns)

# The dataframes are converted to numpy arrays to allow for the matrix math that is going to be involved
X = X_standardized.to_numpy()
y = y_standardized.to_numpy()

# This represents the observation to fold split for the k fold operations
k_N_Splits = observations_N // k_fold 

# Random is once again used to give us a random initial beta, -1 and 1 are used as a range
initial_beta = np.random.uniform(-1, 1, feature_total_p)

# Variable to keep track of all beta results for graphing
record_of_v_beta = []

# Variable to keep track of the cost history for graphing
record_of_cost = []

# The iteration loop size is set to 100,000 this time around
iter_loops = 5_000

# b_k_set for coordinate descent calculations
b_k_set = np.sum(X**2, axis=0)
beta_loop_length = len(initial_beta)

def coordinate_descent(X_training_set, y_training_set, initial_beta, tuning_parameter, elastic_tp, iter_loops, beta_loop_length, b_k_set):
    """
    Perform coordinate descent for the elastic net.
    """
    intermediary_beta = initial_beta.copy()
    
    for iter_c in range(iter_loops):
        for k_current in range(beta_loop_length):
            X_k_current = X_training_set[:, k_current]
            
            # Calculate a_k
            a_k = np.dot(X_k_current, y_training_set - np.dot(X_training_set, intermediary_beta) + X_k_current * intermediary_beta[k_current])
            lambda_adjustment = tuning_parameter * (1 - elastic_tp) / 2
            denominator = b_k_set[k_current] + tuning_parameter * elastic_tp
            
            # Update intermediary_beta[k_current] based on the soft-thresholding rules
            if a_k < -lambda_adjustment:
                intermediary_beta[k_current] = (a_k + lambda_adjustment) / denominator
            elif a_k > lambda_adjustment:
                intermediary_beta[k_current] = (a_k - lambda_adjustment) / denominator
            else:
                intermediary_beta[k_current] = 0
                
    return intermediary_beta


def cross_validation(X, y, tuning_parameter, elastic_tp, k_fold, k_N_Splits, iter_loops, beta_loop_length, b_k_set):
    """
    Perform k-fold cross-validation.
    """
    observations_N = len(y)
    random_reorder = np.random.permutation(observations_N)
    
    cross_validation_account = []  # Track CV errors
    cross_validation_beta_account = []  # Track beta values
    initial_beta = np.random.uniform(-1, 1, X.shape[1])
    
    for fold in range(k_fold):
        # Observation placement after random reorganization
        beg_index = fold * k_N_Splits
        end_index = (fold + 1) * k_N_Splits
        validation_unit = random_reorder[beg_index:end_index]

        # Validation set
        X_validation_set = X[validation_unit]
        y_validation_set = y[validation_unit]

        # Training set
        X_training_set = np.delete(X, validation_unit, axis=0)
        y_training_set = np.delete(y, validation_unit, axis=0)

        # Perform coordinate descent to update beta coefficients
        intermediary_beta = coordinate_descent(X_training_set, y_training_set, initial_beta, tuning_parameter, elastic_tp, iter_loops, beta_loop_length, b_k_set)

        # Predict validation set output
        y_validation_set_predicted = np.dot(X_validation_set, intermediary_beta)
        # Calculate validation error
        validation_error = np.sum((y_validation_set - y_validation_set_predicted) ** 2) / len(y_validation_set)

        # Append validation error and beta values for this fold
        cross_validation_account.append(validation_error)
        cross_validation_beta_account.append(intermediary_beta.copy())

    return np.mean(cross_validation_account), np.mean(cross_validation_beta_account, axis=0)


def tuning_cv_loop_elastic(tuning_parameters_set, elastic_tp, X, y, k_fold, k_N_Splits, iter_loops, beta_loop_length, b_k_set):
    """
    Main loop to tune parameters with cross-validation for Elastic Net.
    """
    cross_validation_account_final = []
    vector_parameter_beta_final = []
    cross_validation_beta_account_final = []

    for current_tuning_parameter in tuning_parameters_set:
        cross_validation_error, mean_beta = cross_validation(X, y, current_tuning_parameter, elastic_tp, k_fold, k_N_Splits, iter_loops, beta_loop_length, b_k_set)
        
        cross_validation_account_final.append(cross_validation_error)
        vector_parameter_beta_final.append(mean_beta)
        cross_validation_beta_account_final.append(mean_beta)  # This could be adjusted based on your requirements

    print(cross_validation_account_final)
    return cross_validation_account_final, vector_parameter_beta_final, cross_validation_beta_account_final

# Elastic tuning parameter values
elastic_tuning_parameter_set = [0, .2, .4, .6, .8, 1]
etp_cross_validation_account_final = []
etp_vector_parameter_beta_final = []
etp_cross_validation_beta_account_final = []

for elastic_tuning_parameter in elastic_tuning_parameter_set:
    elastic_tp = elastic_tuning_parameter
    cross_validation_account_final, vector_parameter_beta_final, cross_validation_beta_account_final = tuning_cv_loop_elastic(
        tuning_parameters_set, elastic_tp, X, y, k_fold, k_N_Splits, iter_loops, beta_loop_length, b_k_set
    )
    etp_cross_validation_account_final.append(cross_validation_account_final)
    etp_vector_parameter_beta_final.append(vector_parameter_beta_final)
    etp_cross_validation_beta_account_final.append(cross_validation_beta_account_final)

# Graphing Functions
def coefficients_vp_tp(current_vector_param_b, graph_title):
    current_vector_param_b = np.array(current_vector_param_b)
    
    tp_log_set = [-2, -1, 0, 1, 2, 3, 4, 5, 6]

    plt.figure(figsize=(10, 6))

    # Using the beta values found at each penalty parameter a graph is made. Labels show a beta for each specific column
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 0], label="Income")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 1], label="Limit")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 2], label="Rating")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 3], label="Cards")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 4], label="Education")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 5], label="Gender")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 6], label="Married")
    plt.semilogx(tuning_parameters_set, current_vector_param_b[:, 7], label="Student")
    plt.title(graph_title)
    plt.xlabel("Tuning Parameter")
    plt.ylabel("Coefficient Values")
    plt.legend()
    plt.grid()
    plt.show()

# Final graph visualizations
for index, cross_validation_beta_account in enumerate(etp_cross_validation_beta_account_final):
    coefficients_vp_tp(cross_validation_beta_account, f"Elastic Tuning Parameter: {elastic_tuning_parameter_set[index]}")

# The end of the integrated code



[26561.664873249283, 26450.44313462973, 25705.704803869205, 25592.608280193894, 26356.665628231345, 26743.55932574441, 33052.693688514846, 89035.84259238564, 210849.779775]
[25696.85400218111, 26085.149563775125, 25202.898347792216, 26228.185753723366, 29833.498301775766, 57436.54560216799, 142118.89737240734, 203927.99527972838, 210849.779775]
[25788.536609110077, 26068.418997021792, 25576.350001424427, 27202.757617492724, 33627.42066127054, 75450.10565632608, 167373.63374262938, 206757.79494196628, 210849.779775]
[25850.98377976021, 25671.970975445518, 25933.13478767508, 27468.75683492388, 37383.22979241035, 89842.4532160221, 178985.80152457618, 207742.7743106674, 210849.77977500003]


# __________________________________________________________________
# __________________________________________________________________

In [2]:
# __________________________________________________________________
# __________________________________________________________________